In [8]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression,Lasso,Ridge
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
import warnings

In [9]:
df = pd.read_csv(r'Data/StudentsPerformance.csv')
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [10]:
x = df.drop(columns = ['math score'])
x.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75


In [11]:
y = df['math score']
y.head()

0    72
1    69
2    90
3    47
4    76
Name: math score, dtype: int64

In [12]:
# To segregate cat features and num features into different sets
num_features = x.select_dtypes(exclude = 'object').columns
cat_features = x.select_dtypes(include = 'object').columns

num_transformer = StandardScaler()
cat_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [('OneHotEncoder', cat_transformer, cat_features),
     ('StandardScaler', num_transformer, num_features)]
)


In [13]:
X = preprocessor.fit_transform(x)

In [14]:
X.shape

(1000, 19)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
X_train.shape, X_test.shape

((800, 19), (200, 19))

In [16]:
def model_evaluation(actual, prediction):
    MAE = mean_absolute_error(actual, prediction)
    R_squared = r2_score(actual, prediction)
    MSE = mean_squared_error(actual, prediction)
    RMSE = np.sqrt(MSE)
    return MAE, R_squared, RMSE

In [21]:
models = {
    'Linear Regression':LinearRegression(),
    'Ridge':Ridge(),
    'Lasso':Lasso(),
    'KNN': KNeighborsRegressor(),
    'DT': DecisionTreeRegressor(),
    'RandomForest': RandomForestRegressor(),
    'AdaBoost':AdaBoostRegressor(),
    'CatBoost':CatBoostRegressor(verbose = False),
    'XgbRegressor':XGBRegressor()
}

models_list = []
r2_list = []

for i in range(len(models)):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae, model_train_Rsquare, model_train_rmse = model_evaluation(y_train, y_train_pred)
    model_test_mae, model_test_Rsquare, model_test_rmse = model_evaluation(y_test, y_test_pred)

    print(list(models.keys())[i])
    models_list.append(list(models.keys())[i])
    r2_list.append(model_test_Rsquare)

    print("Model performance on training set")
    print('- COD value: {:.4f}'.format(model_train_Rsquare))
    print('- MAE value: {:.4f}'.format(model_train_mae))
    print('- RMSE value: {:.4f}'.format(model_train_rmse))

    print('-------------------------------')

    print("Model performance on test set")
    print('- COD value: {:.4f}'.format(model_test_Rsquare))
    print('- MAE value: {:.4f}'.format(model_test_mae))
    print('- RMSE value: {:.4f}'.format(model_test_rmse))

    print("="*20)
          
    

Linear Regression
Model performance on training set
- COD value: 0.8743
- MAE value: 4.2671
- RMSE value: 5.3244
-------------------------------
Model performance on test set
- COD value: 0.8803
- MAE value: 4.2158
- RMSE value: 5.3960
Ridge
Model performance on training set
- COD value: 0.8743
- MAE value: 4.2650
- RMSE value: 5.3233
-------------------------------
Model performance on test set
- COD value: 0.8806
- MAE value: 4.2111
- RMSE value: 5.3904
Lasso
Model performance on training set
- COD value: 0.8071
- MAE value: 5.2063
- RMSE value: 6.5938
-------------------------------
Model performance on test set
- COD value: 0.8253
- MAE value: 5.1579
- RMSE value: 6.5197
KNN
Model performance on training set
- COD value: 0.8555
- MAE value: 4.5167
- RMSE value: 5.7077
-------------------------------
Model performance on test set
- COD value: 0.7838
- MAE value: 5.6210
- RMSE value: 7.2530
DT
Model performance on training set
- COD value: 0.9997
- MAE value: 0.0187
- RMSE value: 0.2

In [22]:
pd.DataFrame(list(zip(models_list, r2_list)), columns = ['Model Name', 'COD']).sort_values(by = ['COD'], ascending = False)

,Model Name,COD
1,Ridge,0.880593
0,Linear Regression,0.880345
7,CatBoost,0.851632
5,RandomForest,0.849722
6,AdaBoost,0.848363
8,XgbRegressor,0.827797
2,Lasso,0.825320
3,KNN,0.783813
4,DT,0.736684
